# Session 29 Solutions: Pipelines

Data Science with Python, Level 2, Stage 4: Applied ML

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])

## In-Class Exercises

### In-Class Exercise 1: Spot the Leak

In [2]:
answers = {
    "a": "leaks: the scaler saw the test rows",
    "b": "safe",
    "c": "mostly safe: dropping rows does not learn from the target, but keep it consistent for new data",
    "d": "leaks: feature choice used the test rows' targets",
}
for k, v in answers.items():
    print(k, v)

a leaks: the scaler saw the test rows
b safe
c mostly safe: dropping rows does not learn from the target, but keep it consistent for new data
d leaks: feature choice used the test rows' targets


### In-Class Exercise 2: Three Models, One Pipeline

In [3]:
from sklearn.ensemble import RandomForestClassifier
num_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
cat_steps = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore"))
preprocess = ColumnTransformer([
    ("num", num_steps, make_column_selector(dtype_include="number")),
    ("cat", cat_steps, make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
pipe = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
for m in [LogisticRegression(max_iter=1000), RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=0, n_jobs=-1), HistGradientBoostingClassifier(learning_rate=0.05, max_depth=3, random_state=0)]:
    pipe.set_params(model=m)
    print(type(m).__name__, cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean().round(3))

LogisticRegression 0.787
RandomForestClassifier 0.774
HistGradientBoostingClassifier 0.771


### In-Class Exercise 3: Log Charges

In [4]:
num_log = make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler())
cat_steps = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore"))
pre_log = ColumnTransformer([
    ("num", num_log, make_column_selector(dtype_include="number")),
    ("cat", cat_steps, make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
p = Pipeline([("prep", pre_log), ("model", LogisticRegression(max_iter=1000))])
print(cross_val_score(p, X, y, cv=5, scoring="roc_auc").mean().round(3))

0.782


### In-Class Exercise 4: A Learning Transformer

In [5]:
class ClipOutliers(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.low_ = X.quantile(0.01)
        self.high_ = X.quantile(0.99)
        return self

    def transform(self, X):
        return X.clip(self.low_, self.high_, axis=1)

clip = ClipOutliers().fit(X[["monthly_charges", "total_charges"]])
print(clip.low_.round(1).to_dict(), clip.high_.round(1).to_dict())
clip.transform(X[["monthly_charges", "total_charges"]]).describe().round(1)

{'monthly_charges': 18.0, 'total_charges': 65.3} {'monthly_charges': 102.9, 'total_charges': 6022.4}


,monthly_charges,total_charges
count,2000.0,1989.0
mean,66.0,1872.9
std,22.3,1388.5
min,18.0,65.3
25%,51.9,802.0
50%,69.6,1556.5
75%,83.6,2627.6
max,102.9,6022.4


## Part A: Basic

### Exercise 1: Name the Steps

In [6]:
p = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression())])
print(p.named_steps.keys())
p.set_params(clf__C=0.1)
print(p["clf"].C)

dict_keys(['scale', 'clf'])
0.1


### Exercise 2: Columns by Type

In [7]:
print(make_column_selector(dtype_include="number")(X))
print(make_column_selector(dtype_include=["object", "string"])(X))

['tenure_months', 'monthly_charges', 'total_charges', 'support_calls']
['gender', 'contract', 'internet', 'payment_method']


## Part B: Intermediate

### Exercise 3: Apartment Price Pipeline

In [8]:
from sklearn.linear_model import Ridge
flats = pd.read_csv(DATA_URL + "apartments.csv")
Xa = flats.drop(columns=["apartment_id", "price_k"])
pre = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
p = Pipeline([("prep", pre), ("model", Ridge())])
print(-cross_val_score(p, Xa, flats["price_k"], cv=5, scoring="neg_mean_absolute_error").mean().round(1))

10.0


### Exercise 4: Price per Square Metre

In [9]:
from sklearn.linear_model import Ridge
flats = pd.read_csv(DATA_URL + "apartments.csv")
Xa = flats.drop(columns=["apartment_id", "price_k"])
class AddAreaFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["area_per_room"] = X["area_m2"] / X["rooms"]
        X["old_building"] = X["building_age"] > 30
        return X
pre = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
for steps in [[("prep", pre), ("model", Ridge())], [("feat", AddAreaFeatures()), ("prep", pre), ("model", Ridge())]]:
    p = Pipeline(steps)
    print(len(steps), -cross_val_score(p, Xa, flats["price_k"], cv=5, scoring="neg_mean_absolute_error").mean().round(2))

2 9.98
3 10.0


## Part C: Challenge

### Exercise 5: A Target Encoder Without Leakage

scikit-learn 1.3 and later includes `TargetEncoder`, which does this with extra protection against overfitting.

In [10]:
class MeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column
    def fit(self, X, y):
        self.means_ = pd.Series(y.values, index=X.index).groupby(X[self.column]).mean()
        self.overall_ = float(np.mean(y))
        return self
    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].map(self.means_).astype(float).fillna(self.overall_)
        return X

cols = ["payment_method", "tenure_months", "monthly_charges"]
p = make_pipeline(MeanEncoder("payment_method"), StandardScaler(), LogisticRegression())
print(cross_val_score(p, X[cols], y, cv=5, scoring="roc_auc").mean().round(3))
# The encoder learns from the target. Fitting it outside the pipeline would leak test targets into training.

0.73


## Part D: Cumulative Project: Customer Churn

### Exercise 6: Customer Churn: Step 6, The Final Pipeline Design

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
X_dev, X_final, y_dev, y_final = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
class AddChurnFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["calls_per_year"] = X["support_calls"] / (X["tenure_months"] / 12 + 1)
        X["new_customer"] = X["tenure_months"] < 6
        X["charges_per_tenure"] = X["monthly_charges"] / (X["tenure_months"] + 1)
        return X
def preprocess():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include=["object", "string"])),
        ("bool", "passthrough", make_column_selector(dtype_include="bool")),
    ])
options = {
    "logistic, no features": Pipeline([("prep", preprocess()), ("model", LogisticRegression(max_iter=1000))]),
    "logistic, features": Pipeline([("features", AddChurnFeatures()), ("prep", preprocess()), ("model", LogisticRegression(max_iter=1000))]),
    "boosting, features": Pipeline([("features", AddChurnFeatures()), ("prep", preprocess()), ("model", HistGradientBoostingClassifier(learning_rate=0.05, max_depth=3, random_state=0))]),
}
for name, p in options.items():
    print(name, cross_val_score(p, X_dev, y_dev, cv=5, scoring="roc_auc").mean().round(3))

logistic, no features 0.786
logistic, features 0.785
boosting, features 0.778


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | It makes scores look better than reality. |
| 2 | B | Each fold must learn its own scaling. |
| 3 | B | Step name, two underscores, setting. |
| 4 | A | TransformerMixin adds `fit_transform` from them. |
| 5 | B | That is why the pipeline is what you deploy. |